In [1]:
import numpy as np
import trimesh
import os, glob

from panel_designer import panel

In [2]:
# resolution = '_coarse'
pattern = 'Imedco_finecircle_update'
meshstrings = ['-x','-y','+x','+y']
coilmeshes = [
    pattern+'-x.stl',
    pattern+'-y.stl',
    pattern+'+x.stl',
    pattern+'+y.stl'
]

contour_file = 'Imedco_update_n12_G10_mumetal.txt'

In [3]:
planes = []
for stl in coilmeshes:
    planes.append(
        trimesh.load(
            file_obj= os.path.relpath(stl),
            process=False,
            validate=True
        )
    )

# Generate a bunch of planes

## Load wires and mesh objects into memory

First, load all wires back into a numpy array

In [4]:
contours = []
with open(contour_file) as f:
    wire = f.readline()
    while wire:
        wire = wire.replace('\n','').replace(';',',').split(',')
        raw = [float(xyz) for xyz in wire]
        contours.append(np.reshape(raw,(-1,3)))
        wire = f.readline()
contours = np.array(contours,dtype=object)

Next, find which mesh object corresponds to which wire. Do this by finding the closest distance from each point on the wire to each mesh object, and take the mesh object that's closest. This is an expensive operation, so it may take a few minutes to run.

To help things run faster, it decimates the contour fed into the proximity.closest_point function

```python
    contour[::4]
```

In [5]:
contour_address = []
plane_r = np.zeros(len(planes))
for contour in contours:
    for i,plane in enumerate(planes):
        _,r,_ =trimesh.proximity.closest_point(plane,contour[::4])
        plane_r[i]=np.mean(r)
    closest = np.reshape(np.argwhere(plane_r==plane_r.min()),(1,-1))[0]
    contour_address.append(closest[0])
    assert len(closest)==1, "There should only be one closest mesh, not %d" % len(closest)

contour_address = np.array(contour_address,dtype=int)

## Generate a new file for each plane

In [10]:
for j,(plane,label) in enumerate(zip(planes,meshstrings)):
    
    nx = np.unique(plane.vertices[:,0])
    ny = np.unique(plane.vertices[:,1])
    nz = np.unique(plane.vertices[:,2])
    
    paper_x_sign = 1
    
    if len(nx)==1:
        xi = 1
        yj = 2
        paper_x_sign = np.sign(nx[0])
    elif len(ny)==1:
        xi = 0
        yj = 2
        paper_x_sign = -1*np.sign(ny[0])
    elif len(nz)==1:
        xi=0
        yj=1
        paper_x_sign = np.sign(nz[0])
    else:
        assert True, "Mesh object is not simple plane. Might need to transform or unwrap more carefully."
    
    path=plane.outline()
    frame = []
    for line in path.entities:
        frame.append(
            [[x*paper_x_sign,y] for x,y in plane.vertices[line.points][:,(xi,yj)]]
        )
        
    wires = contours[contour_address==j]
        
    DXF_loops = []
    for loop in wires:
        temp = [[x*paper_x_sign, y] for x,y in loop[:,(xi,yj)]]
        if len(temp)>4: #don't try to draw or connect nubbins
            temp.append(temp[0]) # close loop
            DXF_loops.append(np.array(temp).T)

    thisface = panel(DXF_loops)
    thisface.make_dxf('imedcotest+framed'+label,
                      frame = frame,
                      gap=0.001,
                      length=0.002,
                      stride=4,
                      textstring='G10 ImedcoCircle '+label+f' mumetal 2025.10.18',
                      textcenter=np.array((0, 0)),
                      textheight=0.01,)
    